In [83]:
import mysql.connector
from pymongo import MongoClient

db_mysql = mysql.connector.connect(
    host="localhost",
    user="root", 
    password="", 
    database="saludusm" 
)
cursor_mysql = db_mysql.cursor(dictionary=True)

uri_mongo = "mongodb://mongo1:30001,mongo2:30002,mongo3:30003/?replicaSet=my-replica-set&readPreference=primary"
cliente_mongo = MongoClient(uri_mongo)
db_mongo = cliente_mongo["saludusm"]


### Migración de sql


In [84]:
#MIGRACIÓN DE ESPECIALIDADES 

print("--- Migrando Especialidades ---")
# Extraer de MySQL
cursor_mysql.execute("SELECT id_espec, tipo FROM Especialidades")
especialidades_sql = cursor_mysql.fetchall()

# Transformar a JSON (Diccionarios)
docs_especialidades = []
for fila in especialidades_sql:
    docs_especialidades.append({
        "_id": fila["id_espec"], 
        "nombre": fila["tipo"]
    })

# Insertar en MongoDB
db_mongo.especialidades.delete_many({}) # Limpiar por si se ejecuta dos veces
if docs_especialidades:
    db_mongo.especialidades.insert_many(docs_especialidades)

# Evidencia obligatoria
print(f"Total en MySQL: {len(especialidades_sql)}")
print(f"Total en MongoDB: {db_mongo.especialidades.count_documents({})}\n")

# MIGRACIÓN DE CENTROS MÉDICOS

print("--- Migrando Centros Médicos ---")
# Extraer de MySQL
cursor_mysql.execute("SELECT id_cm, nombre_cm, comuna, region FROM centros_medicos")
centros_sql = cursor_mysql.fetchall()

# Transformar a JSON
docs_centros = []
for fila in centros_sql:
    docs_centros.append({
        "_id": fila["id_cm"], 
        "nombre": fila["nombre_cm"],
        "comuna": fila["comuna"],
        "region": fila["region"]
    })

# Insertar en MongoDB
db_mongo.centros_medicos.delete_many({})
if docs_centros:
    db_mongo.centros_medicos.insert_many(docs_centros)

# Evidencia obligatoria
print(f"Total en MySQL: {len(centros_sql)}")
print(f"Total en MongoDB: {db_mongo.centros_medicos.count_documents({})}\n")

# MIGRACIÓN DE PACIENTES

print("--- Migrando Pacientes ---")
cursor_mysql.execute("SELECT rut_pac, id_prevision, Nombre_pac, fecha_de_nacimiento, sexo, telefono_de_contacto FROM pacientes")
pacientes_sql = cursor_mysql.fetchall()

docs_pacientes = []
for fila in pacientes_sql:
    docs_pacientes.append({
        "_id": fila["rut_pac"],
        "nombre": fila["Nombre_pac"],
        "fecha_nacimiento": str(fila["fecha_de_nacimiento"]), # Convertido a string por compatibilidad de formato
        "sexo": fila["sexo"],
        "telefono": fila["telefono_de_contacto"],
        "id_prevision": fila["id_prevision"] 
    })

db_mongo.pacientes.delete_many({})
if docs_pacientes:
    db_mongo.pacientes.insert_many(docs_pacientes)

print(f"Total en MySQL: {len(pacientes_sql)}")
print(f"Total en MongoDB: {db_mongo.pacientes.count_documents({})}\n")

# MIGRACIÓN DE MÉDICOS

print("--- Migrando Médicos ---")
cursor_mysql.execute("SELECT rut_med, Nombre_med, Email FROM medicos")
medicos_sql = cursor_mysql.fetchall()

docs_medicos = []
for medico in medicos_sql:
    rut = medico["rut_med"]
    cursor_mysql.execute(f"SELECT id_espec FROM medico_especialidad WHERE rut_med = '{rut}'")
    ids_especialidades = [f["id_espec"] for f in cursor_mysql.fetchall()]
        
    cursor_mysql.execute(f"SELECT id_cm FROM medicos_centro WHERE rut_med = '{rut}'")
    ids_centros = [f["id_cm"] for f in cursor_mysql.fetchall()]
    
    docs_medicos.append({
        "_id": rut,
        "nombre": medico["Nombre_med"],
        "correo": medico["Email"],
        "ids_especialidades": ids_especialidades, 
        "ids_centros": ids_centros 
    })
    
# Estas líneas ahora están fuera del ciclo 'for'
db_mongo.medicos.delete_many({})
if docs_medicos:
    db_mongo.medicos.insert_many(docs_medicos)
    
print(f"Total en MySQL: {len(medicos_sql)}")
print(f"Total en MongoDB: {db_mongo.medicos.count_documents({})}\n")


# MIGRACIÓN DE PREVISIONES
print("--- Migrando Previsiones de Salud ---")
# Extraer de MySQL (Ajustando a los nombres exactos de tu tabla)
cursor_mysql.execute("SELECT id_prevision, Nombre_prevision FROM Prevision_de_salud")
previsiones_sql = cursor_mysql.fetchall()

# Transformar a JSON (Diccionarios)
docs_previsiones = []
for fila in previsiones_sql:
    docs_previsiones.append({
        "_id": fila["id_prevision"], 
        "nombre": fila["Nombre_prevision"]
    })

# Insertar en MongoDB
db_mongo.previsiones.delete_many({}) 
if docs_previsiones:
    db_mongo.previsiones.insert_many(docs_previsiones)

# Evidencia 
print(f"Total Previsiones en MySQL: {len(previsiones_sql)}")
print(f"Total Previsiones en MongoDB: {db_mongo.previsiones.count_documents({})}\n")

# MIGRACIÓN DE ESTADOS DE CITA
print("--- Migrando Estados de Cita ---")
cursor_mysql.execute("SELECT id_estado, tipo_estado FROM Estado")
estados_sql = cursor_mysql.fetchall()

docs_estados = []
for fila in estados_sql:
    docs_estados.append({
        "_id": fila["id_estado"], 
        "nombre": fila["tipo_estado"]
    })

db_mongo.estados_cita.delete_many({})
if docs_estados:
    db_mongo.estados_cita.insert_many(docs_estados)

print(f"Total Estados en MySQL: {len(estados_sql)}")
print(f"Total en MongoDB: {db_mongo.estados_cita.count_documents({})}\n")


# MIGRACIÓN DE CITAS Y ATENCIONES
print("--- Migrando Citas y Atenciones ---")
cursor_mysql.execute("SELECT id_cita, rut_pac, rut_med, id_cm, id_espec, id_estado, fecha_y_hora, comentario FROM Citas")
citas_sql = cursor_mysql.fetchall()

docs_citas = []
for cita in citas_sql:
    id_cita = cita["id_cita"]
    
    fecha_hora_obj = cita["fecha_y_hora"]
    fecha_str = fecha_hora_obj.strftime("%Y-%m-%d")
    hora_str = fecha_hora_obj.strftime("%H:%M")

    documento_cita = {
        "_id": f"CITA{id_cita:03d}", 
        "fecha": fecha_str,
        "hora": hora_str,
        "rut_paciente": cita["rut_pac"],
        "rut_medico": cita["rut_med"],
        "codigo_centro": cita["id_cm"],
        "id_especialidad": cita["id_espec"],
        "id_estado": cita["id_estado"]
    }
    
    cursor_mysql.execute(f"SELECT id_atencion, motivo, observaciones FROM Atenciones WHERE id_cita = {id_cita}")
    atencion_sql = cursor_mysql.fetchone() 
    
    if atencion_sql:
        id_atencion = atencion_sql["id_atencion"]
        
        consulta_diagnosticos = f"""
            SELECT d.id_diag, d.descripcion 
            FROM Diagnosticos d
            JOIN Diagnosticos_atenciones da ON d.id_diag = da.id_diag
            WHERE da.id_atencion = {id_atencion}
        """
        cursor_mysql.execute(consulta_diagnosticos)
        diagnosticos_formateados = [{"codigo_cie10": d["id_diag"], "descripcion": d["descripcion"]} for d in cursor_mysql.fetchall()]
        
        cursor_mysql.execute(f"SELECT medicamento, dosis, dias_tratamiento FROM Recetas WHERE id_atencion = {id_atencion}")
        recetas_sql = cursor_mysql.fetchall()
        
        documento_cita["atencion"] = {
            "motivo": atencion_sql["motivo"],
            "observaciones": atencion_sql["observaciones"],
            "diagnosticos": diagnosticos_formateados, 
            "recetas": recetas_sql 
        }
    
    docs_citas.append(documento_cita)

db_mongo.citas.delete_many({})
if docs_citas:
    db_mongo.citas.insert_many(docs_citas)

print(f"Total Citas en MySQL: {len(citas_sql)}")
print(f"Total Citas en MongoDB: {db_mongo.citas.count_documents({})}\n")
    

--- Migrando Especialidades ---
Total en MySQL: 8
Total en MongoDB: 8

--- Migrando Centros Médicos ---
Total en MySQL: 4
Total en MongoDB: 4

--- Migrando Pacientes ---
Total en MySQL: 200
Total en MongoDB: 200

--- Migrando Médicos ---
Total en MySQL: 50
Total en MongoDB: 50

--- Migrando Previsiones de Salud ---
Total Previsiones en MySQL: 3
Total Previsiones en MongoDB: 3

--- Migrando Estados de Cita ---
Total Estados en MySQL: 5
Total en MongoDB: 5

--- Migrando Citas y Atenciones ---
Total Citas en MySQL: 600
Total Citas en MongoDB: 600



### Consulta a: Agenda diaria de un médico

In [85]:
rut_medico = "12000001-1" 
fecha_consulta = "2026-05-23" 

pipeline_a = [
    {"$match": {"rut_medico": rut_medico, "fecha": fecha_consulta}},
    # JOIN con Pacientes
    {"$lookup": {"from": "pacientes", "localField": "rut_paciente", "foreignField": "_id", "as": "paciente"}},
    {"$unwind": "$paciente"},
    # JOIN con Previsiones (desde paciente)
    {"$lookup": {"from": "previsiones", "localField": "paciente.id_prevision", "foreignField": "_id", "as": "prevision"}},
    {"$unwind": "$prevision"},
    # JOIN con Centros Médicos
    {"$lookup": {"from": "centros_medicos", "localField": "codigo_centro", "foreignField": "_id", "as": "centro"}},
    {"$unwind": "$centro"},
    # JOIN con Especialidades
    {"$lookup": {"from": "especialidades", "localField": "id_especialidad", "foreignField": "_id", "as": "especialidad"}},
    {"$unwind": "$especialidad"},
    # JOIN con Estados
    {"$lookup": {"from": "estados_cita", "localField": "id_estado", "foreignField": "_id", "as": "estado"}},
    {"$unwind": "$estado"},
    # Ordenar por hora y Proyectar campos finales
    {"$sort": {"hora": 1}},
    {"$project": {
        "_id": 0, "hora": 1,
        "paciente": "$paciente.nombre", "prevision": "$prevision.nombre",
        "centro": "$centro.nombre", "especialidad": "$especialidad.nombre", "estado": "$estado.nombre"
    }}
]

resultados_a = list(db_mongo.citas.aggregate(pipeline_a))
for r in resultados_a:
    print(f"{r['hora']} | Pct: {r['paciente']} | Prev: {r['prevision']} | Centro: {r['centro']} | Esp: {r['especialidad']} | Est: {r['estado']}")
if not resultados_a: 
    print("Sin citas para este día.")

23:00 | Pct: Benjamín Fuentes Oyarzo | Prev: Isapre | Centro: CM Las Condes | Esp: Cardiología | Est: Atendida


**Explicación:**  
Se utiliza el pipeline de agregación filtrando inicialmente (`$match`) por el RUT del médico y la fecha consultada[cite: 7]. Luego, se emplean múltiples `$lookup` y `$unwind` para cruzar la información con las colecciones referenciadas (pacientes, previsiones, centros, especialidades y estados) y obtener sus nombres. Finalmente, se ordena (`$sort`) por hora de forma ascendente y se proyectan (`$project`) únicamente los campos solicitados para la visualización[cite: 7].

**Consulta SQL equivalente (Tarea 2):**
```sql
SELECT c.fecha_y_hora, p.Nombre_pac, pr.Nombre_prevision, cm.nombre_cm, e.tipo, es.tipo_estado
FROM Citas c
JOIN Pacientes p ON c.rut_pac = p.rut_pac
JOIN Prevision_de_salud pr ON p.id_prevision = pr.id_prevision
JOIN Centros_medicos cm ON c.id_cm = cm.id_cm
JOIN Especialidades e ON c.id_espec = e.id_espec
JOIN Estado es ON c.id_estado = es.id_estado
WHERE c.rut_med = '12000001-1' AND DATE(c.fecha_y_hora) = '2023-10-25'
ORDER BY TIME(c.fecha_y_hora) ASC;

### Consulta b: Historial clínico de un paciente.

In [86]:
rut_paciente = "22000001-1" 

pipeline_b = [
    {"$match": {"rut_paciente": rut_paciente, "atencion": {"$exists": True}}},
    {"$lookup": {"from": "centros_medicos", "localField": "codigo_centro", "foreignField": "_id", "as": "centro"}},
    {"$unwind": "$centro"},
    {"$lookup": {"from": "especialidades", "localField": "id_especialidad", "foreignField": "_id", "as": "especialidad"}},
    {"$unwind": "$especialidad"},
    {"$lookup": {"from": "medicos", "localField": "rut_medico", "foreignField": "_id", "as": "medico"}},
    {"$unwind": "$medico"},
    {"$sort": {"fecha": -1, "hora": -1}},
    {"$project": {
        "_id": 0, "fecha": 1, "centro": "$centro.nombre", "especialidad": "$especialidad.nombre",
        "medico": "$medico.nombre", "motivo": "$atencion.motivo",
        "diagnosticos": "$atencion.diagnosticos", "recetas": "$atencion.recetas"
    }}
]

resultados_b = list(db_mongo.citas.aggregate(pipeline_b))

for r in resultados_b:
    print(f"\nFecha: {r['fecha']} | {r['centro']} | {r['especialidad']} | Dr. {r['medico']}")
    print(f"Motivo: {r['motivo']}")
    nombres_diag = [d['descripcion'] for d in r['diagnosticos']]
    detalle_recetas = [f"{m['medicamento']} ({m['dosis']})" for m in r['recetas']]
    
    print(f"Diagnósticos: {nombres_diag}")
    print(f"Recetas: {detalle_recetas}")


Fecha: 2026-08-02 | CM Curauma | Cardiología | Dr. Tomás Contreras Herrera
Motivo: Control de presión
Diagnósticos: ['Hipotiroidismo', 'Diabetes mellitus tipo 2']
Recetas: ['Salbutamol (1 al día en ayunas)']


**Explicación:**  
Se busca por el RUT del paciente exigiendo con `$exists` que el documento contenga el subdocumento `atencion` (filtrando así solo citas atendidas)[cite: 7]. Como el modelo documental ya embebe diagnósticos y recetas dentro de la cita, solo se requiere hacer `$lookup` a las colecciones maestras (centro, especialidad y médico) para obtener sus descriptores. El resultado se ordena cronológicamente de forma descendente[cite: 7].

**Consulta SQL equivalente (Tarea 2):**
```sql
SELECT c.fecha_y_hora, cm.nombre_cm, e.tipo, m.Nombre_med, a.motivo
FROM Citas c
JOIN Atenciones a ON c.id_cita = a.id_cita
JOIN Centros_medicos cm ON c.id_cm = cm.id_cm
JOIN Especialidades e ON c.id_espec = e.id_espec
JOIN Medicos m ON c.rut_med = m.rut_med
WHERE c.rut_pac = '22000001-1'
ORDER BY c.fecha_y_hora DESC;

### Consulta c: Tasa de inasistencia por centro.

In [87]:
pipeline_c = [
    {"$lookup": {"from": "estados_cita", "localField": "id_estado", "foreignField": "_id", "as": "estado"}},
    {"$unwind": "$estado"},
    {"$lookup": {"from": "centros_medicos", "localField": "codigo_centro", "foreignField": "_id", "as": "centro"}},
    {"$unwind": "$centro"},
    {"$group": {
        "_id": "$centro.nombre",
        "total_citas": {"$sum": 1},
        "inasistencias": {"$sum": {"$cond": [{"$eq": ["$estado.nombre", "No Asistió"]}, 1, 0]}}
    }},
    {"$project": {
        "total_citas": 1, "inasistencias": 1,
        "porcentaje": {"$round": [{"$multiply": [{"$divide": ["$inasistencias", "$total_citas"]}, 100]}, 2]}
    }},
    {"$match": {"porcentaje": {"$gt": 15}}},
    {"$sort": {"porcentaje": -1}}
]

for r in db_mongo.citas.aggregate(pipeline_c):
    print(f"Centro: {r['_id']} | Total: {r['total_citas']} | Inasistencias: {r['inasistencias']} | Tasa: {r['porcentaje']}%")

Centro: CM Viña Centro | Total: 142 | Inasistencias: 29 | Tasa: 20.42%


**Explicación:**  
Se emplea `$group` por el nombre del centro para obtener el `total_citas` usando una sumatoria simple, mientras que las inasistencias se cuentan mediante un operador condicional `$cond` que evalúa si el estado es "No Asistió"[cite: 7]. En la fase `$project`, se realiza la matemática del porcentaje redondeando a 2 decimales (`$round`). Finalmente, un `$match` filtra solo aquellos centros con tasa mayor al 15% y los ordena de forma descendente[cite: 7].

**Consulta SQL equivalente (Tarea 2):**
```sql
SELECT cm.nombre_cm, COUNT(c.id_cita) as total, 
SUM(IF(es.tipo_estado = 'No Asistió', 1, 0)) as inasistencias,
ROUND((SUM(IF(es.tipo_estado = 'No Asistió', 1, 0)) / COUNT(c.id_cita)) * 100, 2) as porcentaje
FROM Citas c
JOIN Centros_medicos cm ON c.id_cm = cm.id_cm
JOIN Estado es ON c.id_estado = es.id_estado
GROUP BY cm.id_cm
HAVING porcentaje > 15
ORDER BY porcentaje DESC;

### Consulta d: Ranking de medicamentos por especialidad.

In [88]:
especialidad_buscada = "Pediatría" 

pipeline_d = [
    {"$lookup": {"from": "especialidades", "localField": "id_especialidad", "foreignField": "_id", "as": "especialidad"}},
    {"$unwind": "$especialidad"},
    {"$match": {"especialidad.nombre": especialidad_buscada, "atencion.recetas": {"$exists": True}}},
    {"$unwind": "$atencion.recetas"},
    {"$group": {
        "_id": "$atencion.recetas.medicamento",
        "cantidad": {"$sum": 1}
    }},
    {"$sort": {"cantidad": -1, "_id": 1}},
    {"$limit": 5}
]

for i, r in enumerate(db_mongo.citas.aggregate(pipeline_d), 1):
    print(f"{i}. {r['_id']} - {r['cantidad']} recetas")

1. Salbutamol - 9 recetas
2. Amoxicilina - 8 recetas
3. Loratadina - 8 recetas
4. Omeprazol - 8 recetas
5. Ibuprofeno - 4 recetas


**Explicación:**  
Se busca por el nombre de la especialidad y se exige que el arreglo de recetas exista. Utilizando `$unwind`, se desanida el arreglo `atencion.recetas` generando un flujo de documentos individuales por cada medicamento. Luego se agrupan (`$group`) por su nombre para sumarlos, se ordenan de mayor a menor y se limitan a los primeros 5 resultados (`$limit`)[cite: 7]. Como criterio de desempate en `$sort`, se incluye el nombre del medicamento (`_id: 1`) alfabéticamente[cite: 7].

**Consulta SQL equivalente (Tarea 2):**
```sql
SELECT r.medicamento, COUNT(*) as cantidad
FROM Recetas r
JOIN Atenciones a ON r.id_atencion = a.id_atencion
JOIN Citas c ON a.id_cita = c.id_cita
JOIN Especialidades e ON c.id_espec = e.id_espec
WHERE e.tipo = 'Pediatría'
GROUP BY r.medicamento
ORDER BY cantidad DESC, r.medicamento ASC
LIMIT 5;

### Consulta e: Pacientes multi-especialidad.


In [89]:
pipeline_e = [
    {"$lookup": {"from": "estados_cita", "localField": "id_estado", "foreignField": "_id", "as": "estado"}},
    {"$unwind": "$estado"},
    {"$match": {"estado.nombre": "Atendida"}}, 
    {"$lookup": {"from": "especialidades", "localField": "id_especialidad", "foreignField": "_id", "as": "especialidad"}},
    {"$unwind": "$especialidad"},
    {"$group": {
        "_id": "$rut_paciente",
        "especialidades_unicas": {"$addToSet": "$especialidad.nombre"}
    }},
    {"$project": {
        "cantidad_especialidades": {"$size": "$especialidades_unicas"},
        "lista_especialidades": "$especialidades_unicas"
    }},
    {"$match": {"cantidad_especialidades": {"$gte": 3}}},
    {"$lookup": {"from": "pacientes", "localField": "_id", "foreignField": "_id", "as": "paciente"}},
    {"$unwind": "$paciente"},
    {"$sort": {"cantidad_especialidades": -1}},
    {"$project": {
        "_id": 0, "paciente": "$paciente.nombre", 
        "cantidad": "$cantidad_especialidades", "detalle": "$lista_especialidades"
    }}
]

for r in db_mongo.citas.aggregate(pipeline_e):
    print(f"Paciente: {r['paciente']} | Total Especialidades: {r['cantidad']}")
    print(f"   -> {', '.join(r['detalle'])}")

Paciente: Catalina Villarroel Valenzuela | Total Especialidades: 5
   -> Psiquiatría, Dermatología, Traumatología, Cardiología, Pediatría
Paciente: Martina Herrera Oyarzo | Total Especialidades: 4
   -> Neurología, Cardiología, Psiquiatría, Oftalmología
Paciente: Noah Morales Contreras | Total Especialidades: 4
   -> Medicina General, Oftalmología, Dermatología, Neurología
Paciente: Lucas Alarcón Muñoz | Total Especialidades: 4
   -> Traumatología, Dermatología, Psiquiatría, Cardiología
Paciente: Catalina Morales Valenzuela | Total Especialidades: 4
   -> Pediatría, Traumatología, Oftalmología, Medicina General
Paciente: Ethan Oyarzo Villarroel | Total Especialidades: 4
   -> Neurología, Cardiología, Traumatología, Dermatología
Paciente: Mateo Soto Valenzuela | Total Especialidades: 4
   -> Medicina General, Neurología, Traumatología, Oftalmología
Paciente: Isabella Morales Reyes | Total Especialidades: 4
   -> Psiquiatría, Dermatología, Traumatología, Cardiología
Paciente: Isabella Al

**Explicación:**  
Tras filtrar únicamente las citas "Atendidas", se agrupa por el RUT del paciente. La clave aquí es el operador `$addToSet`, el cual recopila los nombres de las especialidades ignorando los duplicados. Luego se emplea `$size` para contar la longitud de este arreglo de especialidades únicas, se filtra usando `$match` para retener solo los pacientes con 3 o más, y se ordena por cantidad.

**Consulta SQL equivalente (Tarea 2):**
```sql
SELECT p.Nombre_pac, COUNT(DISTINCT c.id_espec) as total_especialidades
FROM Citas c
JOIN Pacientes p ON c.rut_pac = p.rut_pac
JOIN Estado es ON c.id_estado = es.id_estado
WHERE es.tipo_estado = 'Atendida'
GROUP BY p.rut_pac
HAVING total_especialidades >= 3
ORDER BY total_especialidades DESC;

### Consistencia de los datos

In [90]:
from pymongo import MongoClient, WriteConcern
from datetime import datetime
from pymongo.errors import ServerSelectionTimeoutError

# Conexión principal con Write Concern Majority
uri_principal = "mongodb://mongo1:30001,mongo2:30002,mongo3:30003/?replicaSet=my-replica-set"
cliente_principal = MongoClient(uri_principal)
db_principal = cliente_principal["saludusm"]
citas_majority = db_principal.get_collection("citas", write_concern=WriteConcern(w="majority"))

# Configurar conexiones directas
cliente_n1 = MongoClient("mongodb://mongo1:30001/?directConnection=true&serverSelectionTimeoutMS=2000")
cliente_n2 = MongoClient("mongodb://mongo2:30002/?directConnection=true&serverSelectionTimeoutMS=2000")
cliente_n3 = MongoClient("mongodb://mongo3:30003/?directConnection=true&serverSelectionTimeoutMS=2000")

def verificar_en_nodos(filtro, mensaje):
    print(f"\nVerificando consistencia tras: {mensaje}")
    for i, cliente_nodo in enumerate([cliente_n1, cliente_n2, cliente_n3], 1):
        try:
            db_nodo = cliente_nodo["saludusm"]
            cantidad = db_nodo.citas.count_documents(filtro)
            print(f" -> Nodo {i}: {cantidad} documento(s) encontrado(s).")
        except ServerSelectionTimeoutError:
            print(f" -> Nodo {i}: [Inaccesible o Apagado]")
        except Exception as e:
            print(f" -> Nodo {i}: Error inesperado ({e})")

# Operación A: Inserción de nueva cita (Reservada)

estado_reservada = db_principal.estados_cita.find_one({"nombre": "Reservada"})
id_cita_prueba = "CITA_PRUEBA_999"

citas_majority.delete_one({"_id": id_cita_prueba}) # Limpiar ejecuciones previas

if estado_reservada:
    nueva_cita = {
        "_id": id_cita_prueba,
        "fecha": "2026-12-31",
        "hora": "10:00",
        "rut_paciente": "11111111-1",
        "id_estado": estado_reservada["_id"]
    }
    citas_majority.insert_one(nueva_cita)
    verificar_en_nodos({"_id": id_cita_prueba}, "Insertar cita en estado 'Reservada'")

# Operación B: Actualización de citas vencidas a 'Cancelada'

estado_confirmada = db_principal.estados_cita.find_one({"nombre": "Confirmada"})
estado_cancelada = db_principal.estados_cita.find_one({"nombre": "Cancelada"}) 

if estado_cancelada and estado_reservada:
    ids_activos = [estado_reservada["_id"]]
    if estado_confirmada: 
        ids_activos.append(estado_confirmada["_id"])
        
    fecha_actual = datetime.now().strftime("%Y-%m-%d")
    filtro_upd = {
        "fecha": {"$lt": fecha_actual},
        "id_estado": {"$in": ids_activos}
    }
    
    citas_majority.update_many(filtro_upd, {"$set": {"id_estado": estado_cancelada["_id"]}})
    verificar_en_nodos({"id_estado": estado_cancelada["_id"]}, f"Actualizar citas antiguas a 'Cancelada' (Fecha de corte: {fecha_actual})")

# Operación C: Eliminación de una línea de receta

cita_con_receta = db_principal.citas.find_one({"atencion.recetas.0": {"$exists": True}})

if cita_con_receta:
    id_cita_modificar = cita_con_receta["_id"]
    medicamento_a_borrar = cita_con_receta["atencion"]["recetas"][0]["medicamento"]
    
    # Operador $pull para eliminar un elemento específico del arreglo
    citas_majority.update_one(
        {"_id": id_cita_modificar}, 
        {"$pull": {"atencion.recetas": {"medicamento": medicamento_a_borrar}}}
    )
    
    filtro_receta = {
        "_id": id_cita_modificar, 
        "atencion.recetas.medicamento": medicamento_a_borrar
    }
    print(f"\nVerificando consistencia tras: Eliminar línea de receta ({medicamento_a_borrar}) en {id_cita_modificar}")
    for i, cliente_nodo in enumerate([cliente_n1, cliente_n2, cliente_n3], 1):
        try:
            cantidad = cliente_nodo["saludusm"].citas.count_documents(filtro_receta)
            print(f" -> Nodo {i}: {cantidad} documento(s) encontrado(s) con esa receta (Esperado: 0).")
        except ServerSelectionTimeoutError:
            print(f" -> Nodo {i}: [Inaccesible o Apagado]")


Verificando consistencia tras: Insertar cita en estado 'Reservada'
 -> Nodo 1: 1 documento(s) encontrado(s).
 -> Nodo 2: 1 documento(s) encontrado(s).
 -> Nodo 3: 1 documento(s) encontrado(s).

Verificando consistencia tras: Actualizar citas antiguas a 'Cancelada' (Fecha de corte: 2026-11-03)
 -> Nodo 1: 55 documento(s) encontrado(s).
 -> Nodo 2: 55 documento(s) encontrado(s).
 -> Nodo 3: 55 documento(s) encontrado(s).

Verificando consistencia tras: Eliminar línea de receta (Salbutamol) en CITA001
 -> Nodo 1: 0 documento(s) encontrado(s) con esa receta (Esperado: 0).
 -> Nodo 2: 0 documento(s) encontrado(s) con esa receta (Esperado: 0).
 -> Nodo 3: 0 documento(s) encontrado(s) con esa receta (Esperado: 0).


### 4.4 Consistencia de Datos: Actualización Automática de Estados

**Explicación:**  
Para mantener la consistencia de la base de datos, se ejecuta un script de mantenimiento que busca todas las citas cuya fecha ya ocurrió (es estrictamente menor `<` a la fecha actual `$lt`) y que, por error u omisión, conservan un estado activo (`$in` ["Reservada", "Confirmada"]). Mediante la operación `update_many()` y el operador `$set`, el motor de MongoDB actualiza masivamente el `id_estado` de todos los documentos que coinciden con el filtro, asignándoles el ID correspondiente a "No Asistió" sin necesidad de procesarlos uno por uno en Python.

**Consulta SQL equivalente (Tarea 2):**
```sql
UPDATE Citas 
SET id_estado = (SELECT id_estado FROM Estado WHERE tipo_estado = 'No Asistió')
WHERE fecha_y_hora < CURDATE() 
  AND id_estado IN (
      SELECT id_estado FROM Estado WHERE tipo_estado IN ('Reservada', 'Confirmada')
  );

### Nodo principal

In [91]:
print(db_principal.command("isMaster")['primary'])

mongo2:30002


### Prueba de alta disponibilidad

In [92]:
# Se utiliza el mismo cliente genérico del replica set
id_fallo = "CITA_FALLO_123"

# Limpieza previa
db_principal.citas.delete_one({"_id": id_fallo})

try:
    resultado = db_principal.citas.insert_one({
        "_id": id_fallo,
        "nota": "Prueba de Alta Disponibilidad",
        "fecha": "2026-12-31"
    })
    print(f"¡ÉXITO! La cita se insertó con ID: {resultado.inserted_id}")
    
    # Comprobar lectura inmediata
    leida = db_principal.citas.find_one({"_id": id_fallo})
    if leida:
        print(f"Datos leídos correctamente: {leida}")
    
except ServerSelectionTimeoutError:
    print("Fallo crítico: El clúster completo está inaccesible o no hay nodos suficientes para formar quórum.")
except Exception as e:
    print(f"Fallo en la operación: {e}")

¡ÉXITO! La cita se insertó con ID: CITA_FALLO_123
Datos leídos correctamente: {'_id': 'CITA_FALLO_123', 'nota': 'Prueba de Alta Disponibilidad', 'fecha': '2026-12-31'}


In [93]:
#Cambiar n1 al nodo que fue apagado

try:
    cita_recuperada = cliente_n1["saludusm"].citas.find_one({"_id": 1111111111})
    if cita_recuperada:
        print(f"¡Éxito! El Nodo se sincronizó automáticamente. Datos: {cita_recuperada}")
    else:
        print("El nodo aún se está sincronizando, intenta ejecutar la celda otra vez en unos segundos.")
except Exception as e:
    print(f"Error de conexión: {e}")

El nodo aún se está sincronizando, intenta ejecutar la celda otra vez en unos segundos.


### Exportación a **JSON**

In [94]:
import os
from bson.json_util import dumps

# Crear carpeta para la entrega
carpeta_salida = "colecciones_json"
os.makedirs(carpeta_salida, exist_ok=True)

# Colecciones de la base de datos
colecciones = db_principal.list_collection_names()

for nombre_col in colecciones:
    # Extraer todos los documentos de la colección
    documentos = list(db_principal[nombre_col].find({}))
    ruta_archivo = os.path.join(carpeta_salida, f"{nombre_col}.json")
    with open(ruta_archivo, 'w', encoding='utf-8') as archivo:
        archivo.write(dumps(documentos, indent=4, ensure_ascii=False))
        
    print(f"Exportada colección '{nombre_col}' -> {len(documentos)} documentos.")

print(f"\n¡Exportación exitosa! Todos los archivos están en la carpeta '{carpeta_salida}'.")

Exportada colección 'centros_medicos' -> 4 documentos.
Exportada colección 'especialidades' -> 8 documentos.
Exportada colección 'medicos' -> 50 documentos.
Exportada colección 'estados_cita' -> 5 documentos.
Exportada colección 'previsiones' -> 3 documentos.
Exportada colección 'pacientes' -> 200 documentos.
Exportada colección 'citas' -> 602 documentos.

¡Exportación exitosa! Todos los archivos están en la carpeta 'colecciones_json'.
